# 🔄 Data Pipelines & Workflow Orchestration — Interactive Notebook

**Author:** Youssef Ibrahim Mohamed Soliman  
**GitHub:** https://github.com/Yosef-Ibrahim  
**Email:** youssefibrahimelisely@gmail.com  
**Phone:** 01119834356  

---  
This interactive notebook demonstrates custom DAG orchestration engine mechanics: task registration, dependency resolution, topological sorting, and XCom data passing.

In [ ]:
class CustomDAG:
    def __init__(self, dag_id):
        self.dag_id = dag_id
        self.tasks = {}
        self.dependencies = {}
        self.xcom_store = {}

    def add_task(self, task_id, func):
        self.tasks[task_id] = func
        self.dependencies[task_id] = set()

    def set_dependency(self, upstream_task_id, downstream_task_id):
        if downstream_task_id in self.dependencies:
            self.dependencies[downstream_task_id].add(upstream_task_id)

    def execute(self):
        print(f"[+] Starting Execution for DAG: '{self.dag_id}'")
        completed = set()
        while len(completed) < len(self.tasks):
            progress_made = False
            for task_id, func in self.tasks.items():
                if task_id not in completed:
                    upstream_tasks = self.dependencies[task_id]
                    if upstream_tasks.issubset(completed):
                        print(f"  --> Executing Task: [{task_id}] (Upstream satisfied: {list(upstream_tasks)})")
                        result = func(self.xcom_store)
                        if result is not None:
                            self.xcom_store[task_id] = result
                        completed.add(task_id)
                        progress_made = True
            if not progress_made:
                print("❌ Deadlock Detected!")
                break
        print(f"[SUCCESS] DAG '{self.dag_id}' Completed!")

In [ ]:
dag = CustomDAG("umbrella_demand_pipeline")

def fetch_weather(xcom):
    return {"city": "Cairo", "temp": 32.5}

def fetch_sales(xcom):
    return {"umbrella_units": 150}

def combine_data(xcom):
    return {"merged_rows": 1000}

dag.add_task("fetch_weather", fetch_weather)
dag.add_task("fetch_sales", fetch_sales)
dag.add_task("combine_data", combine_data)

dag.set_dependency("fetch_weather", "combine_data")
dag.set_dependency("fetch_sales", "combine_data")

dag.execute()